# Problem Statement #1: Name Classifier

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

# Load the dataset
file_path = r'c:\Users\raghu\H-518\Assignment-5\name_gender.csv'
data = pd.read_csv(file_path)

# Remove non-ASCII characters from names
data['name'] = data['name'].apply(lambda x: ''.join([char for char in x if char.isascii()]))

# Encode gender labels
label_encoder = LabelEncoder()
data['gender'] = label_encoder.fit_transform(data['gender'])  # 0 for Female, 1 for Male

# Display the first few rows of the dataset
print(data.head())

        name  gender  probability
0      Aaban       1          1.0
1      Aabha       0          1.0
2      Aabid       1          1.0
3  Aabriella       0          1.0
4       Aada       0          1.0


In [28]:
class NameDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.max_length = max(data['name'].apply(len))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        name = self.data.iloc[idx]['name']
        gender = self.data.iloc[idx]['gender']
        name_tensor = torch.zeros(self.max_length, dtype=torch.long)
        for i, char in enumerate(name):
            name_tensor[i] = ord(char)
        return name_tensor, torch.tensor(gender, dtype=torch.long)

In [27]:
percentages = [0.25, 0.50, 0.75, 1.00]

datasets = {}

for percentage in percentages:
    sampled_data = data.sample(frac=percentage, random_state=42)
    
    train_data, test_data = train_test_split(sampled_data, test_size=0.2, random_state=42)
    
    datasets[percentage] = {'train': train_data, 'test': test_data}

In [36]:
for percentage in percentages:
    print(len(datasets[percentage]['train']), "training samples for percentage:", percentage)

19004 training samples for percentage: 0.25
38010 training samples for percentage: 0.5
57016 training samples for percentage: 0.75
76020 training samples for percentage: 1.0


### Simple RNN

In [38]:
class NameClassifierRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NameClassifierRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(128, input_size)  # ASCII values range from 0 to 127
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        output = self.fc(hidden.squeeze(0))
        return output

In [39]:
input_size = 16
hidden_size = 64
output_size = 2
model = NameClassifierRNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
num_epochs = 5

In [41]:
models = {}

for percentage in percentages:
    # Prepare the dataset split
    train_data, _ = train_test_split(datasets[percentage]['train'], test_size=0.2, random_state=42)
    train_dataset = NameDataset(train_data)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Initialize a new model for each percentage
    model = NameClassifierRNN(input_size, hidden_size, output_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    # Train the model
    print(f"Training model with {percentage * 100}% of the dataset...")
    for epoch in range(num_epochs):
        for names, genders in train_loader:
            optimizer.zero_grad()
            outputs = model(names)
            loss = criterion(outputs, genders)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}")

    # Save the trained model
    models[f"model_{int(percentage * 100)}"] = model

Training model with 25.0% of the dataset...
Epoch 1/5, Loss: 0.9792
Epoch 2/5, Loss: 0.2489
Epoch 3/5, Loss: 0.9151
Epoch 4/5, Loss: 0.4967
Epoch 5/5, Loss: 0.8964
Training model with 50.0% of the dataset...
Epoch 1/5, Loss: 0.5913
Epoch 2/5, Loss: 0.1920
Epoch 3/5, Loss: 0.1117
Epoch 4/5, Loss: 0.1254
Epoch 5/5, Loss: 0.4469
Training model with 75.0% of the dataset...
Epoch 1/5, Loss: 0.4762
Epoch 2/5, Loss: 0.2126
Epoch 3/5, Loss: 0.4343
Epoch 4/5, Loss: 0.5833
Epoch 5/5, Loss: 0.4074
Training model with 100.0% of the dataset...
Epoch 1/5, Loss: 0.7514
Epoch 2/5, Loss: 0.0783
Epoch 3/5, Loss: 0.6913
Epoch 4/5, Loss: 0.5494
Epoch 5/5, Loss: 0.2086


### LSTM

In [42]:
class NameClassifierLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NameClassifierLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(128, input_size)
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        output = self.fc(hidden.squeeze(0))
        return output

In [44]:
models_LSTM = {}
num_epochs_LSTM = 5  # Number of epochs for LSTM training

for percentage in percentages:
    # Prepare the dataset split
    train_data, _ = train_test_split(datasets[percentage]['train'], test_size=0.2, random_state=42)
    train_dataset = NameDataset(train_data)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Initialize a new LSTM model for each percentage
    model_LSTM = NameClassifierLSTM(input_size, hidden_size, output_size)
    criterion_LSTM = nn.CrossEntropyLoss()
    optimizer_LSTM = optim.Adam(model_LSTM.parameters(), lr=0.0005)

    # Train the LSTM model
    print(f"Training LSTM model with {percentage * 100}% of the dataset...")
    for epoch in range(num_epochs_LSTM):
        for names, genders in train_loader:
            optimizer_LSTM.zero_grad()
            outputs = model_LSTM(names)
            loss = criterion_LSTM(outputs, genders)
            loss.backward()
            optimizer_LSTM.step()
        print(f"Epoch {epoch + 1}/{num_epochs_LSTM}, Loss: {loss.item():.4f}")

    # Save the trained LSTM model
    models_LSTM[f"model_LSTM_{int(percentage * 100)}"] = model_LSTM

Training LSTM model with 25.0% of the dataset...
Epoch 1/5, Loss: 0.3397
Epoch 2/5, Loss: 0.7546
Epoch 3/5, Loss: 0.1481
Epoch 4/5, Loss: 0.1454
Epoch 5/5, Loss: 0.0990
Training LSTM model with 50.0% of the dataset...
Epoch 1/5, Loss: 0.1200
Epoch 2/5, Loss: 0.1688
Epoch 3/5, Loss: 0.3042
Epoch 4/5, Loss: 0.1800
Epoch 5/5, Loss: 0.6690
Training LSTM model with 75.0% of the dataset...
Epoch 1/5, Loss: 0.5259
Epoch 2/5, Loss: 0.5729
Epoch 3/5, Loss: 0.1847
Epoch 4/5, Loss: 0.2361
Epoch 5/5, Loss: 0.1330
Training LSTM model with 100.0% of the dataset...
Epoch 1/5, Loss: 0.5321
Epoch 2/5, Loss: 0.2618
Epoch 3/5, Loss: 0.2540
Epoch 4/5, Loss: 0.3393
Epoch 5/5, Loss: 0.3587


### GRU

In [45]:
class NameClassifierGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NameClassifierGRU, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(128, input_size)  # ASCII values range from 0 to 127
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        output = self.fc(hidden.squeeze(0))
        return output

In [46]:
models_GRU = {}
num_epochs_GRU = 5 

for percentage in percentages:
    # Prepare the dataset split
    train_data, _ = train_test_split(datasets[percentage]['train'], test_size=0.2, random_state=42)
    train_dataset = NameDataset(train_data)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Initialize a new GRU model for each percentage
    model_GRU = NameClassifierGRU(input_size, hidden_size, output_size)
    criterion_GRU = nn.CrossEntropyLoss()
    optimizer_GRU = optim.Adam(model_GRU.parameters(), lr=0.0005)

    # Train the GRU model
    print(f"Training GRU model with {percentage * 100}% of the dataset...")
    for epoch in range(num_epochs_GRU):
        for names, genders in train_loader:
            optimizer_GRU.zero_grad()
            outputs = model_GRU(names)
            loss = criterion_GRU(outputs, genders)
            loss.backward()
            optimizer_GRU.step()
        print(f"Epoch {epoch + 1}/{num_epochs_GRU}, Loss: {loss.item():.4f}")

    # Save the trained GRU model
    models_GRU[f"model_GRU_{int(percentage * 100)}"] = model_GRU

Training GRU model with 25.0% of the dataset...
Epoch 1/5, Loss: 0.2416
Epoch 2/5, Loss: 0.1562
Epoch 3/5, Loss: 0.5158
Epoch 4/5, Loss: 0.0322
Epoch 5/5, Loss: 0.1826
Training GRU model with 50.0% of the dataset...
Epoch 1/5, Loss: 0.5708
Epoch 2/5, Loss: 0.1507
Epoch 3/5, Loss: 0.4128
Epoch 4/5, Loss: 0.0262
Epoch 5/5, Loss: 0.4596
Training GRU model with 75.0% of the dataset...
Epoch 1/5, Loss: 0.4131
Epoch 2/5, Loss: 0.1316
Epoch 3/5, Loss: 0.1682
Epoch 4/5, Loss: 0.2476
Epoch 5/5, Loss: 0.3353
Training GRU model with 100.0% of the dataset...
Epoch 1/5, Loss: 0.1251
Epoch 2/5, Loss: 0.4841
Epoch 3/5, Loss: 0.2352
Epoch 4/5, Loss: 0.4747
Epoch 5/5, Loss: 0.3328


In [55]:
def calculate_accuracy(model, dataloader):
    model.eval()
    total = 0
    correct = 0
    class_correct = [0] * output_size
    class_total = [0] * output_size

    with torch.no_grad():
        for names, genders in dataloader:
            outputs = model(names)
            _, predicted = torch.max(outputs, 1)
            total += genders.size(0)
            correct += (predicted == genders).sum().item()

            for i in range(len(genders)):
                label = genders[i].item()
                class_total[label] += 1
                class_correct[label] += (predicted[i] == label).item()

    overall_accuracy = correct / total
    classwise_accuracy = [class_correct[i] / class_total[i] if class_total[i] > 0 else 0 for i in range(output_size)]

    return overall_accuracy, classwise_accuracy

In [56]:
report = []

for percentage, dataset_split in datasets.items():
    test_dataset = NameDataset(dataset_split['test'])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    rnn_model = models[f"model_{int(percentage * 100)}"]
    overall_accuracy_rnn, classwise_accuracy_rnn = calculate_accuracy(rnn_model, test_loader)

    lstm_model = models_LSTM[f"model_LSTM_{int(percentage * 100)}"]
    overall_accuracy_lstm, classwise_accuracy_lstm = calculate_accuracy(lstm_model, test_loader)

    gru_model = models_GRU[f"model_GRU_{int(percentage * 100)}"]
    overall_accuracy_gru, classwise_accuracy_gru = calculate_accuracy(gru_model, test_loader)

    report.append({
        "Model": "RNN",
        "Dataset Size": percentage * 100,
        "Overall Accuracy": overall_accuracy_rnn * 100,
        "Male Accuracy": classwise_accuracy_rnn[1] * 100,
        "Female Accuracy": classwise_accuracy_rnn[0] * 100
    })
    report.append({
        "Model": "LSTM",
        "Dataset Size": percentage * 100,
        "Overall Accuracy": overall_accuracy_lstm * 100,
        "Male Accuracy": classwise_accuracy_lstm[1] * 100,
        "Female Accuracy": classwise_accuracy_lstm[0] * 100
    })
    report.append({
        "Model": "GRU",
        "Dataset Size": percentage * 100,
        "Overall Accuracy": overall_accuracy_gru * 100,
        "Male Accuracy": classwise_accuracy_gru[1] * 100,
        "Female Accuracy": classwise_accuracy_gru[0] * 100
    })

report_df = pd.DataFrame(report)
report_df = report_df.sort_values(by="Overall Accuracy", ascending=True)
print(report_df)

   Model  Dataset Size  Overall Accuracy  Male Accuracy  Female Accuracy
0    RNN          25.0         82.218013      85.388662        80.434068
1   LSTM          25.0         83.964646      81.355932        85.432424
2    GRU          25.0         84.785354      80.245470        87.339691
3    RNN          50.0         84.994212      87.588753        83.467068
6    RNN          75.0         85.155044      81.008274        87.534504
4   LSTM          50.0         85.667684      82.391366        87.596122
9    RNN         100.0         85.751868      82.934132        87.399933
5    GRU          50.0         85.983374      82.703777        87.913741
7   LSTM          75.0         86.017960      80.527227        89.168599
8    GRU          75.0         86.333661      85.183760        86.993486
10  LSTM         100.0         86.441124      83.561449        88.125417
11   GRU         100.0         86.972535      83.575706        88.959306


## Observations 

1. Overall Accuracy Trends: Accuracy improves as dataset size increases, with GRU performing the best at 100.0 dataset size (86.97%).
2. Model Performance: GRU generally achieves the highest overall accuracy, followed by LSTM and then RNN.
3. Gender-Based Accuracy: Male accuracy is highest with RNN at 50.0 dataset size (87.59%), while female accuracy peaks with LSTM at 75.0 dataset size (89.17%).
4. Trade-offs: RNN performs well with smaller datasets, but GRU and LSTM offer better scalability and balance in male and female accuracy.

# Problem Statement #2: Train a language model using these names.

In [162]:
class NameGeneratorRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NameGeneratorRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        output = self.fc(output)
        return output, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size)

In [163]:
print(data.head(5))

        name  gender  probability
0      Aaban       1          1.0
1      Aabha       0          1.0
2      Aabid       1          1.0
3  Aabriella       0          1.0
4       Aada       0          1.0


In [164]:
input_size = 128  # ASCII values range from 0 to 127
hidden_size = 128
output_size = 128  # Same as input_size for character-level generation

language_model = NameGeneratorRNN(input_size, hidden_size, output_size)

In [171]:
class NameDataset(Dataset):
    def __init__(self, names, all_chars):
        self.names = names
        self.char_to_idx = {char: i for i, char in enumerate(all_chars)}
        self.idx_to_char = {i: char for i, char in enumerate(all_chars)}
        self.all_chars = all_chars

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        # Create input and target sequences for training
        input_seq = [self.char_to_idx.get(c, 0) for c in name[:-1]]
        target_seq = [self.char_to_idx.get(c, 0) for c in name[1:]]

        return (
            torch.tensor(input_seq, dtype=torch.long),
            torch.tensor(target_seq, dtype=torch.long)
        )

In [169]:
from torch.nn.utils.rnn import pad_sequence

all_chars = ''.join([chr(i) for i in range(128)])
names = data['name'].tolist()
language_dataset = NameDataset(names, all_chars)

# Create a DataLoader with collate_fn to pad sequences
def collate_fn(batch):
    inputs, targets = zip(*batch)
    padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=0)
    padded_targets = pad_sequence(targets, batch_first=True, padding_value=0)
    return padded_inputs, padded_targets

language_loader = DataLoader(language_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [172]:
# Training the NameGeneratorRNN model
num_epochs = 10
learning_rate = 0.002
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(language_model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    total_loss = 0
    for inputs, targets in language_loader:
        optimizer.zero_grad()
        hidden = language_model.init_hidden(inputs.size(0))
        outputs, _ = language_model(inputs, hidden)
        outputs = outputs.view(-1, outputs.size(-1))
        targets = targets.view(-1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss:.4f}")

Epoch 1/10, Loss: 3939.9175
Epoch 2/10, Loss: 3752.6665
Epoch 3/10, Loss: 3703.0959
Epoch 4/10, Loss: 3694.8338
Epoch 5/10, Loss: 3663.1200
Epoch 6/10, Loss: 3660.9052
Epoch 7/10, Loss: 3652.2803
Epoch 8/10, Loss: 3640.7852
Epoch 9/10, Loss: 3635.7505
Epoch 10/10, Loss: 3629.3732


In [201]:
import random

def generate_name(model, start_char, max_length=10):
    model.eval()
    generated_name = [ord(start_char)]
    hidden = model.init_hidden(1)
    input_tensor = torch.tensor([ord(start_char)], dtype=torch.long).unsqueeze(0)
    
    for _ in range(max_length - 1):
        output, hidden = model(input_tensor, hidden)
        probabilities = torch.softmax(output[0, -1], dim=0)
        next_char_idx = torch.multinomial(probabilities, 1).item()
        
        # Stop if we generate a non-letter character
        if next_char_idx < ord('A') or (next_char_idx > ord('Z') and next_char_idx < ord('a')) or next_char_idx > ord('z'):
            break
            
        generated_name.append(next_char_idx)
        input_tensor = torch.tensor([[next_char_idx]], dtype=torch.long)

    # Convert ASCII values back to characters and join them
    return ''.join([chr(c) for c in generated_name if chr(c).isalpha()])

# Generate 100 male and 100 female names with random starting characters
male_names = [generate_name(language_model, start_char=random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')) for _ in range(100)]
female_names = [generate_name(language_model, start_char=random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')) for _ in range(100)]

In [202]:
print("Male Names:", male_names)
print("Female Names:", female_names)

Male Names: ['Veryssahnu', 'Yohandania', 'Neroslinan', 'Ohananquez', 'Idaleiasha', 'Karoleenet', 'Yarexetter', 'Kiecqueric', 'Aastentann', 'Judereisha', 'Beauhiritt', 'Breylareas', 'Izaahnidan', 'Kiarhelden', 'Qadiannaha', 'Uzaminesha', 'Quintorahi', 'Lavernalaz', 'Philanedad', 'Esleenahan', 'Vandenafer', 'Vonicalyne', 'Jeffrenaal', 'Uteshawane', 'Zodiriahel', 'Kruthanail', 'Donationna', 'Ibashniesh', 'Vaferalien', 'Zakirahlah', 'Uzinaelyne', 'Charrianje', 'Patianigay', 'Kahlisanes', 'Senajonyet', 'Vioforrotn', 'Gerredaina', 'Lavieneeta', 'Horullyses', 'Zammeerana', 'Xsondiahna', 'Mariounaem', 'Zuylainael', 'Vickananda', 'Yariaghlea', 'Bautlinell', 'Iandraelia', 'Neularisso', 'Hetzhithan', 'Ulthanette', 'Charocynne', 'Anailerani', 'Edelinatel', 'Kiragenque', 'Londalleri', 'Grexxynnee', 'Anjihnahay', 'Anwardolfw', 'Vedneekani', 'Zaviousses', 'Ulleneydan', 'Derrickash', 'Joselianah', 'Brognehlan', 'Radhaneila', 'Milberiase', 'Tylennahne', 'Kirayonnac', 'Onlifeylas', 'Wilseckica', 'Hryand

In [203]:
# Combine male and female names and their actual labels
all_names = male_names + female_names
all_labels = [1] * len(male_names) + [0] * len(female_names)

# Convert names to tensors
max_length = 15  
name_tensors = []
for name in all_names:
    name_tensor = torch.zeros(max_length, dtype=torch.long)
    for i, char in enumerate(name):
        if i < max_length:
            name_tensor[i] = ord(char)
    name_tensors.append(name_tensor)

name_tensors = torch.stack(name_tensors)
model_GRU_100 = models_GRU['model_GRU_100']
model_GRU_100.eval()
with torch.no_grad():
    outputs = model_GRU_100(name_tensors)
    predictions = torch.argmax(outputs, dim=1).tolist()

# Calculate accuracy
correct_predictions = sum([pred == label for pred, label in zip(predictions, all_labels)])
accuracy = correct_predictions / len(all_labels) * 100

print(f"Accuracy of model_GRU_100 on generated names: {accuracy:.2f}%")

Accuracy of model_GRU_100 on generated names: 54.50%


# Problem Statement #2a: Train a language model using names starting with A, M, and Z.

In [204]:
names_with_AMZ = data[data['name'].str.startswith(('A', 'M', 'Z'))]
names_with_AMZ

,name,gender,probability
0,Aaban,1,1.0
1,Aabha,0,1.0
2,Aabid,1,1.0
3,Aabriella,0,1.0
4,Aada,0,1.0
...,...,...,...
95020,Zyvion,1,1.0
95021,Zyvon,1,1.0
95022,Zyyanna,0,1.0
95023,Zyyon,1,1.0


In [234]:

names_with_AMZ_list = names_with_AMZ['name'].tolist()

language_dataset_AMZ = NameDataset(names_with_AMZ_list, all_chars)

language_loader_AMZ = DataLoader(language_dataset_AMZ, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [235]:

all_chars = sorted(set(''.join(names_with_AMZ_list)))
char_to_idx = {char: i for i, char in enumerate(sorted(all_chars))}
idx_to_char = {i: char for char, i in char_to_idx.items()}

input_size = len(all_chars)
output_size = len(all_chars)

In [236]:
len(names_with_AMZ_list)

19080

In [237]:
language_model_AMZ = NameGeneratorRNN(input_size, hidden_size, output_size)

In [238]:
# Training the NameGeneratorRNN model
num_epochs = 10
learning_rate = 0.002
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(language_model_AMZ.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    total_loss = 0
    for inputs, targets in language_loader_AMZ:
        optimizer.zero_grad()
        hidden = language_model_AMZ.init_hidden(inputs.size(0))
        outputs, _ = language_model_AMZ(inputs, hidden)
        outputs = outputs.view(-1, outputs.size(-1))
        targets = targets.view(-1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss:.4f}")

Epoch 1/10, Loss: 808.7797
Epoch 2/10, Loss: 746.2415
Epoch 3/10, Loss: 731.3977
Epoch 4/10, Loss: 721.3724
Epoch 5/10, Loss: 713.6853
Epoch 6/10, Loss: 706.5217
Epoch 7/10, Loss: 707.8193
Epoch 8/10, Loss: 701.5712
Epoch 9/10, Loss: 697.2285
Epoch 10/10, Loss: 694.1425


In [260]:
# Generate 50 names using language_model_AMZ with random starting characters
def generate_name_amz(model, start_char, max_length=10):
	model.eval()
	generated_name = [char_to_idx[start_char]]  # Use char_to_idx to map the character to its index
	hidden = model.init_hidden(1)
	input_tensor = torch.tensor([char_to_idx[start_char]], dtype=torch.long).unsqueeze(0)
	
	for _ in range(max_length - 1):
		output, hidden = model(input_tensor, hidden)
		probabilities = torch.softmax(output[0, -1], dim=0)
		next_char_idx = torch.multinomial(probabilities, 1).item()
		
		# Stop if we generate a padding character (index 0)
		if next_char_idx == 0:
			break
			
		generated_name.append(next_char_idx)
		input_tensor = torch.tensor([[next_char_idx]], dtype=torch.long)

	# Convert indices back to characters using idx_to_char and join them
	return ''.join([idx_to_char[idx] for idx in generated_name])

# Generate 50 names using the fixed generate_name_amz function
generated_names_amz = [generate_name_amz(language_model_AMZ, start_char=random.choice('AMZ')) for _ in range(50)]

In [261]:
# Print the generated names
print("Generated Names:", generated_names_amz)

Generated Names: ['Mitsukarid', 'Zoredianah', 'Aleekoyaha', 'Ardijamere', 'Zforiahnia', 'Zeyanahlae', 'Zfitheemyn', 'Arminaelee', 'Ayleighana', 'Avithaleen', 'Ziaparoeli', 'Zoeiyahnar', 'Anveereyah', 'Margheriah', 'Zephenahia', 'Znieraheah', 'Adelettasi', 'Zikerianah', 'Zadynahama', 'Maudellaha', 'Zakiyahaha', 'Avlenaanah', 'Malisyahah', 'Zachiyahah', 'Anapailena', 'Marloneese', 'Ziaridesto', 'Milasiahab', 'Mashithama', 'Moleneypal', 'Montasynia', 'Amnelianna', 'Zanayleana', 'Mychaellah', 'Aaronnahiy', 'Marnekaela', 'Annisuinmo', 'Zayneyahah', 'Azzeeriush', 'Maryjaston', 'Majuennzed', 'Martraithh', 'Mtovynnees', 'Zandylynne', 'Zerikoshae', 'Mosettenne', 'Zakarahiah', 'Morikairia', 'Mickenarea', 'Zaariannhi']


In [263]:
import math

def calculate_perplexity(model, dataset, char_to_idx):
    model.eval()
    total_loss = 0
    total_length = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, targets in dataset:
            hidden = model.init_hidden(inputs.size(0))
            outputs, _ = model(inputs, hidden)
            outputs = outputs.view(-1, outputs.size(-1))
            targets = targets.view(-1)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * targets.size(0)
            total_length += targets.size(0)

    avg_loss = total_loss / total_length
    perplexity = math.exp(avg_loss)
    return perplexity

# Prepare the dataset for generated names
generated_names_dataset = NameDataset(generated_names_amz, all_chars)
generated_names_loader = DataLoader(generated_names_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# Calculate perplexity for the generated names
perplexity = calculate_perplexity(language_model_AMZ, generated_names_loader, char_to_idx)
print(f"Perplexity of the generated names: {perplexity:.2f}")

Perplexity of the generated names: 5.96


### We got a pretty good score of 5.96 so the model has learned the structure of names and is generating new names effectively.